In [2]:
!pip install pandas numpy faiss-cpu sentence-transformers scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 55.7 MB/s eta 0:00:00


In [3]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/AmazonDataset/7817_1.csv')

In [4]:
# Keep only required columns
df = df[['asins', 'name', 'reviews.text', 'reviews.rating']]

In [5]:
# Drop missing reviews
df = df.dropna(subset=['reviews.text'])

In [6]:
# Reset index
df = df.reset_index(drop=True)

print("Shape:", df.shape)
df.head()

Shape: (1597, 4)


,asins,name,reviews.text,reviews.rating
0,B00QJDU3KY,Kindle Paperwhite,I initially had trouble deciding between the p...,5.0
1,B00QJDU3KY,Kindle Paperwhite,Allow me to preface this with a little history...,5.0
2,B00QJDU3KY,Kindle Paperwhite,I am enjoying it so far. Great for reading. Ha...,4.0
3,B00QJDU3KY,Kindle Paperwhite,I bought one of the first Paperwhites and have...,5.0
4,B00QJDU3KY,Kindle Paperwhite,I have to say upfront - I don't like coroporat...,5.0


In [7]:
df.info()
df.isnull().sum()
df['reviews.rating'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1597 entries, 0 to 1596
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   asins           1597 non-null   object 
 1   name            1597 non-null   object 
 2   reviews.text    1597 non-null   object 
 3   reviews.rating  1177 non-null   float64
dtypes: float64(1), object(3)
memory usage: 50.0+ KB


,count
reviews.rating,
5.0,741
4.0,236
3.0,124
1.0,42
2.0,34


In [10]:
# Clean the text
import re
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text

df['clean_review'] = df['reviews.text'].apply(clean_text)

In [ ]:
"""
1. Vector Representation of Data
Convert each customer review into a vector (embedding) using any embedding model.
This helps represent the meaning of the text numerically.

"""



In [11]:
from sentence_transformers import SentenceTransformer

#Load the model
model=SentenceTransformer('all-MiniLM-L6-v2')

#Convert the reviews into vectors
embeddings=model.encode(df['clean_review'].tolist(),show_progress_bar=True)

print(embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/50 [00:00<?, ?it/s]

(1597, 384)


In [ ]:
"""
2. Introduction to Vector DB
Store all embeddings in a Vector Database (you can use FAISS as a simple implementation).
Each entry should contain:
Review text
Vector embedding

"""


In [14]:
# Store in FAISS (Vector DB)
import faiss
import numpy as np

#Convert the data to numpy Array
embeddings=np.array(embeddings).astype('float32')

# Create Index
dimension=embeddings.shape[1]
index=faiss.IndexFlatL2(dimension)

#Add Vectors to Index
index.add(embeddings)

print("total vectors in index: ",index.ntotal)

total vectors in index:  1597


In [20]:
# Building the search function
def search_reviews(query, k=5):
    # Convert the query into vector
    query_vec=model.encode([query])
    query_vec=np.array(query_vec).astype('float32')

    # Normalize query
    #faiss.normalize_L2(query_vec)

    # Search in FAISS
    distances, indices=index.search(query_vec, k)

    # Get results
    results=df.iloc[indices[0]][['reviews.text','reviews.rating']]
    return results


In [21]:
# Testing the semantic Search
query="battery drains fast"
results = search_reviews(query)

for i, row in results.iterrows():
  print("Review:",row['reviews.text'])
  print("Rating:",row['reviews.rating'])
  print("-"*50)

Review: ..exactly what I wanted, but the battery drains FAST when on hands free mode.
Rating: 4.0
--------------------------------------------------
Review: I've had the Kindle Fire HDX 7 inch tablet for almost 3 years and when I compare it to the Fire HD 10, the earlier tablet comes out the big winner. I find the larger tablet considerably slower. Several apps load quite slowly. The battery is consumed much more quickly. Maybe this is not comparing apples with apples, but I had expected at least the same speed.
Rating: 3.0
--------------------------------------------------
Review: First off,i appreciate you reading this review :) Just so you get a little background on me, we are major tech users. We have multiple tablets from apple,samsung,and asus. This fire tablet is comparable in a different way. This tablet uses a amazon based operating system, Fire os. So no play store or itunes... As a prime member however, this tablet is an incredible value because you get instant access to all

In [25]:
# Keyword Search
def keyword_search(query, k=5):
  results=df[df['clean_review'].str.contains(query.lower(), na=False)]
  return results[['reviews.text','reviews.rating']].head(k)


In [26]:
# Compare semantic search with Keyword Search
query="battery drains fast"
print("Keyword Search Results:")
print(keyword_search(query))

print("\nSemantic Search Results:")
print(search_reviews(query))

Keyword Search Results:
                                          reviews.text  reviews.rating
979  ..exactly what I wanted, but the battery drain...             4.0

Semantic Search Results:
                                           reviews.text  reviews.rating
979   ..exactly what I wanted, but the battery drain...             4.0
244   I've had the Kindle Fire HDX 7 inch tablet for...             3.0
670   First off,i appreciate you reading this review...             4.0
1309  good speaker. 8 hour battery. Good price. Exce...             5.0
556   First, let me say that this is my first Fire t...             5.0
